# PyMAUDE — Quickstart

**PyMAUDE** is a Python library for accessing and analyzing FDA MAUDE (Manufacturer and User Facility Device Experience) adverse event data via a fast [DuckDB](https://duckdb.org/) backend.

This notebook gets you from zero to your first query. For deeper dives into specific capabilities (substring search, enrichment, filtering, trends, raw SQL, archiving), see the notebooks in [`examples/`](./examples).

> **First run:** `add_years(..., download=True)` downloads zip files from the FDA FTP server (~few hundred MB for a few years of device/master/text). Subsequent runs reuse whatever's already downloaded, and skip re-loading files into the database whose content hasn't changed. See the note under [Load data](#2-load) for how to force a fresh download.

---
## 1. Setup

In [ ]:
from pymaude import MaudeDatabase

DB_PATH  = './maude.duckdb'   # persistent DuckDB file
DATA_DIR = './maude_data'     # downloaded zip/txt files live here
YEARS    = '2024-2026'        # adjust to taste; more years = more data

---
## 2. Load data

In [ ]:
db = MaudeDatabase(DB_PATH, data_dir=DATA_DIR, verbose=True, memory_limit='2GB')

db.add_years(
    YEARS,
    tables=['master', 'device', 'text', 'patient'],
    download=True
    # force_download=True
)

**Note:** `download=True` only fetches from FDA if the file isn't already sitting in `DATA_DIR`; it doesn't check whether FDA's copy has changed since you last downloaded it. To force a fresh download (e.g. to pick up mid-month updates to files you already have locally), pass `force_download=True`.

For routine refreshes, `db.update()` is the easier path: it re-requests every table/year you've already loaded and defaults to `force_download=True`, so you don't need to manage that flag by hand:

In [ ]:
db.update(download=True)

---
## 3. Inspect the database

In [ ]:
db.info()

---
## 4. Your first query

`query_device()` does **exact** matching: case-insensitive for `brand_name`, `generic_name`, and `manufacturer_name`, but **case-sensitive** for `product_code` (FDA product codes are always uppercase, e.g. `'NIQ'`).

In [ ]:
# All events for a specific product code (NIQ = venous stents)
niq = db.query_device(product_code='NIQ')
print(f'NIQ (venous stent) events loaded: {len(niq):,}')
niq[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME', 'DATE_RECEIVED']].head(10)

---
## 5. Substring search

`search_by_device_names()` does case-insensitive substring matching across a synthesized `DEVICE_NAME_CONCAT` column — useful since MAUDE entries are often inconsistent about which name field a device shows up in.

In [ ]:
venous_stents = db.search_by_device_names('venous stent')
print(f'Events matching "venous stent": {len(venous_stents):,}')
venous_stents[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'DATE_RECEIVED']]

---
## Next steps

Results are plain pandas DataFrames — export, filter, or plot them however you like. For more, see [`examples/`](./examples):

- **[searching.ipynb](./examples/searching.ipynb)** — OR/AND substring search, grouped search across device classes, event narratives
- **[enrichment_and_filtering.ipynb](./examples/enrichment_and_filtering.ipynb)** — patient outcomes, device & patient problem codes, chained `filter_by_*` methods
- **[trends_and_sql.ipynb](./examples/trends_and_sql.ipynb)** — year-over-year trends, raw SQL access
- **[archiving.ipynb](./examples/archiving.ipynb)** — freezing a reproducible snapshot for publication

In [ ]:
db.close()